# TMC-LM on Google Colab

Trains the **TinyLlama-1.1B-Chat** LoRA for Trinidad Municipal College and outputs a GGUF
model for local Ollama use.

**Pipeline:** sources → dataset → LoRA fine-tune → merge → GGUF (Q4_K_M)

**Runtime:** Free tier T4 GPU. If you are on free tier, keep batch size 1 (default).

**No Google Drive needed:** everything runs in this virtual machine and is saved under
`/content/tmc-llm-artifacts/`. Download the zip (or just the GGUF) at the end.

## 1. Setup

- The repo is cloned fresh into `/content/tmc-llm` (code updates are picked up automatically).
- To train on **your own documents**, add them to `data/raw/tmc_sources/` inside the cloned repo
  (supported: `.txt`, `.md`, `.pdf`, `.docx`, `.xlsx`, `.csv`, `.json`) before running the
  dataset step. The repo already ships with `train.txt`.

- All produced artifacts write to `/content/tmc-llm-artifacts/` (VM-only):
  - `.hf-cache/` - Hugging Face model cache
  - `models/adapters`, `models/merged`, `models/gguf` - training outputs

In [ ]:
import os, pathlib, subprocess, sys

REPO_URL = "https://github.com/jbasilad/tmc-llm"
REPO_DIR = "/content/tmc-llm"

if not os.path.isdir(f"{REPO_DIR}/.git"):
    print(">> Cloning repo")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
os.environ["PYTHONPATH"] = "src"
print("Repo ready at:", os.getcwd())

In [ ]:
ARTIFACTS = "/content/tmc-llm-artifacts"
CACHE_DIR = f"{ARTIFACTS}/.hf-cache"
SOURCES_DIR = "data/raw/tmc_sources"
ADAPTER_DIR = f"{ARTIFACTS}/models/adapters/tmc-lm-tinyllama-lora"
MERGED_DIR = f"{ARTIFACTS}/models/merged/tmc-lm-tinyllama"
GGUF_DIR = f"{ARTIFACTS}/models/gguf"

for d in [CACHE_DIR, ADAPTER_DIR, MERGED_DIR, GGUF_DIR]:
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = CACHE_DIR
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

print("Artifact paths (VM-only):")
print("  HF cache :", CACHE_DIR)
print("  Sources  :", SOURCES_DIR)
print("  Adapters :", ADAPTER_DIR)
print("  Merged   :", MERGED_DIR)
print("  GGUF     :", GGUF_DIR)
print()
print("No Google Drive used. Download artifacts at the end.")

In [ ]:
def run_script(script):
    print(f">> Running {script}")
    log = "/content/run_script.log"
    cmd = f"/bin/bash {script} 2>&1 | tee {log}"
    proc = subprocess.Popen(cmd, shell=True, text=True)
    rc = proc.wait()
    if rc != 0:
        print(f"\n=== {script} FAILED (exit {rc}). Last 40 log lines: ===")
        try:
            lines = open(log, encoding="utf-8", errors="replace").read().splitlines()
            print("\n".join(lines[-40:]))
        except Exception as exc:
            print(f"(could not read log: {exc})")
        raise RuntimeError(f"{script} exited with code {rc} -- see log above")

print(">> Installing ML libraries (torch already present in Colab)")
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "peft", "datasets", "accelerate", "sentencepiece",
    "PyMuPDF", "python-docx", "openpyxl", "pyyaml", "gguf",
], check=True)

print(">> Installing the tmc_llm package")
try:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
    print("   tmc_llm installed as editable package")
except subprocess.CalledProcessError as exc:
    print("   editable install failed, continuing with PYTHONPATH=src")
    print("   (pip exit", exc.returncode, ")")

import torch
device = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print("PyTorch", torch.__version__, "| CUDA:", torch.cuda.is_available(), "| Device:", device)

## 2. Prepare the dataset

Builds `data/processed/{dataset,train,validation,test}.jsonl` from the source documents in
`data/raw/tmc_sources/`.

In [ ]:
os.environ["SOURCE_DIR"] = SOURCES_DIR
os.environ["OUTPUT_DIR"] = "data/processed"
run_script("scripts/prepare_dataset.sh")

In [ ]:
import json
metadata = json.load(open("data/processed/metadata.json", encoding="utf-8"))
print(f"Source files   : {len(metadata['source_files'])}")
print(f"Total examples : {metadata['total_examples']}")
print(f"Train examples : {metadata['train_examples']}")
print(f"Validation     : {metadata['validation_examples']}")
print(f"Test examples  : {metadata['test_examples']}")
print("Sections       :", ", ".join(metadata["sections"]))

## 3. Fine-tune TinyLlama with LoRA

Uses `configs/train_lora.yaml` hyperparameters (fp16, batch size 1, ~80 steps) but sends the
adapter to `/content/tmc-llm-artifacts/models/adapters/`. On a free T4 this takes roughly
10-20 minutes (plus the one-time ~2.2 GB model download on first run).

In [ ]:
import pathlib
import yaml

cfg = yaml.safe_load(pathlib.Path("configs/train_lora.yaml").read_text(encoding="utf-8"))
cfg["output_dir"] = ADAPTER_DIR

colab_cfg = pathlib.Path("configs/train_lora_colab.yaml")
colab_cfg.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")
print(colab_cfg.read_text(encoding="utf-8"))

In [ ]:
os.environ["CONFIG_PATH"] = "configs/train_lora_colab.yaml"
run_script("scripts/train_lora.sh")
print(">> Adapter saved to", ADAPTER_DIR)

## 4. Merge the LoRA adapter into the base model

Produces a full merged model at `/content/tmc-llm-artifacts/models/merged/tmc-lm-tinyllama/`.

In [ ]:
os.environ["BASE_MODEL"] = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
os.environ["ADAPTER_DIR"] = ADAPTER_DIR
os.environ["OUTPUT_DIR"] = MERGED_DIR
run_script("scripts/merge_lora.sh")
print(">> Merged model saved to", MERGED_DIR)

## 5. Convert to GGUF (llama.cpp, built inside Colab)

No Docker needed. The script clones llama.cpp, builds only the `llama-quantize` binary
(CPU-only, portable), converts the merged model to F16, then quantizes to **Q4_K_M**.
First run takes a few extra minutes for the llama.cpp build.

In [ ]:
os.environ["MERGED_MODEL_DIR"] = MERGED_DIR
os.environ["OUTPUT_DIR"] = GGUF_DIR
os.environ["BUILD_JOBS"] = "2"
run_script("scripts/convert_to_gguf.sh")

In [ ]:
import glob, os
for f in sorted(glob.glob(f"{GGUF_DIR}/*.gguf")):
    print(f"{os.path.basename(f):<40} {os.path.getsize(f)/1e6:8.1f} MB")

## 6. Sanity check (in-notebook QA)

Quick test of the **merged** model against a few TMC knowledge questions before you download.

In [ ]:
import json, pathlib, torch
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer

merged_path = pathlib.Path(MERGED_DIR)
print("Merged dir:", merged_path)
if merged_path.exists():
    print("  contents:", sorted(p.name for p in merged_path.iterdir()))
else:
    print("  !! MISSING merged model directory")

cfg_path = merged_path / "config.json"
if cfg_path.exists():
    cfg = json.loads(cfg_path.read_text(encoding="utf-8"))
    print("  config.json model_type:", cfg.get("model_type"))
    if not cfg.get("model_type"):
        print("  >> repairing config.json (missing model_type)")
        base = json.loads(AutoConfig.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0").to_json_string())
        base.update({k: v for k, v in cfg.items() if k != "model_type"})
        cfg_path.write_text(json.dumps(base, indent=2), encoding="utf-8")
        print("  >> repaired")
else:
    print("  !! config.json missing - writing from base model")
    AutoConfig.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0").save_pretrained(MERGED_DIR)

try:
    qa_model = AutoModelForCausalLM.from_pretrained(MERGED_DIR, torch_dtype=torch.float16, device_map="auto")
    qa_tokenizer = AutoTokenizer.from_pretrained(MERGED_DIR)

    SYSTEM = (
        "You are TMC-LM, an offline assistant for Trinidad Municipal College. "
        "Answer using only official TMC knowledge. If the answer is not in the "
        "source, say that the available TMC source does not contain it. "
        "Be concise and professional."
    )

    def format_for_qa(tokenizer, messages):
        if getattr(tokenizer, "chat_template", None):
            return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        user = next(m["content"] for m in messages if m["role"] == "user")
        return f"### User: {user}\n### Assistant:"

    for question in [
        "What is the vision of TMC?",
        "What academic programs does TMC offer?",
        "Give a brief history of TMC.",
    ]:
        prompt = format_for_qa(
            qa_tokenizer,
            [{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}],
        )
        inputs = qa_tokenizer(prompt, return_tensors="pt").to(qa_model.device)
        outputs = qa_model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=True,
            temperature=0.2,
            top_p=0.9,
        )
        answer = qa_tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        print(f"\nQ: {question}\nA: {answer.strip()}\n")
except Exception as exc:
    print(f"\nWARNING: sanity check failed ({type(exc).__name__}: {exc}) - continuing.")
    print("Your GGUF model is still ready for download below.")

## 7. Download your model

Everything lives in the VM under `/content/tmc-llm-artifacts/`. The cell below zips the GGUF
and the LoRA adapter into `/content/tmc-llm-download.zip` for easy downloading from the Colab
file browser (folder icon on the left).

Main files:
- `/content/tmc-llm-artifacts/models/gguf/tmc-lm-tinyllama-q4_k_m.gguf` - **for Ollama**
- `/content/tmc-llm-artifacts/models/adapters/` - LoRA adapter (optional backup)

Then import into local Ollama on Windows: `scripts/create_ollama_model.ps1` + `ollama run tmc-lm`.

> **Session timeout warning:** free Colab sessions shut down after long idle periods and the
> VM is wiped. Download the zip before you close the notebook, or the trained model is lost.

In [ ]:
import glob, os, pathlib, shutil

stage = "/content/tmc-llm-download"
pathlib.Path(stage).mkdir(parents=True, exist_ok=True)
shutil.copytree(GGUF_DIR, f"{stage}/gguf", dirs_exist_ok=True)
shutil.copytree(ADAPTER_DIR, f"{stage}/adapters", dirs_exist_ok=True)

zip_path = "/content/tmc-llm-download.zip"
shutil.make_archive(zip_path[:-4], "zip", root_dir=stage)
print("Created:", zip_path, f"({os.path.getsize(zip_path)/1e6:.1f} MB)")

print("\nGGUF files included:")
for f in sorted(glob.glob(f"{GGUF_DIR}/*.gguf")):
    print("  ", os.path.basename(f), f"({os.path.getsize(f)/1e6:.1f} MB)")

print()
print("Download:")
print("  1) Open the file browser (folder icon on the left)")
print("  2) Navigate to /content/ and right-click tmc-llm-download.zip -> Download")
print("     (or just download tmc-llm-artifacts/models/gguf/tmc-lm-tinyllama-q4_k_m.gguf)")
print()
print("On Windows (inside the tmc-llm repo):")
print("  1. Put the GGUF in  models/gguf/")
print("  2.  scripts/check_gguf.ps1")
print("  3.  scripts/create_ollama_model.ps1")
print("  4.  ollama run tmc-lm")